# Debug queries

Read-only ad hoc queries against the local dbt-duckdb transform file — per CLAUDE.md's
workflow discipline: **investigate here, never hand-patch data.** If something looks
wrong, root-cause it here, then fix it by editing the actual dbt model / script and
adding a test that would have caught it (see plan.md Section 9).

Connects with `read_only=True` so it never conflicts with a concurrent `dbt run`/`dbt test`.
Requires the MinIO container up (`docker compose up -d minio`) since staging models
are views reading straight from raw-zone via httpfs.


In [1]:
import duckdb
con = duckdb.connect('../duckdb_data/takaful_transform.duckdb', read_only=True)
con.sql("LOAD httpfs;")
con.sql("""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_url_style='path';
    SET s3_use_ssl=false;
""")
con.sql("select * from stg_policies limit 10").show()


┌────────────┬────────────────┬──────────────────────────────────┬──────────────────┬─────────┬────────────┬────────────┬────────────┬──────────────┬─────────────────────┬─────────┬──────────┬─────────────────┐
│ policy_id  │ participant_id │           product_name           │ product_category │ has_pif │ start_date │  end_date  │ term_years │ payment_mode │ contribution_amount │ status  │ agent_id │     branch      │
│  varchar   │    varchar     │             varchar              │     varchar      │ boolean │    date    │    date    │   int32    │   varchar    │    decimal(12,2)    │ varchar │ varchar  │     varchar     │
├────────────┼────────────────┼──────────────────────────────────┼──────────────────┼─────────┼────────────┼────────────┼────────────┼──────────────┼─────────────────────┼─────────┼──────────┼─────────────────┤
│ POL0000001 │ PTP001286      │ Family Takaful - Life Protection │ Family           │ true    │ 2021-11-14 │ 2041-11-09 │         20 │ Monthly      │       

## Fund-split invariant check

`gross_amount = prf_amount + pif_amount + wakalah_fee_shareholders_fund` must hold for
every contribution — see CLAUDE.md "Fund-segregation logic". Should return 0 rows.


In [2]:
con.sql("""
    select contribution_id, policy_id, gross_amount, prf_amount, pif_amount,
           wakalah_fee_shareholders_fund,
           gross_amount - (prf_amount + pif_amount + wakalah_fee_shareholders_fund) as diff
    from stg_contributions
    where abs(gross_amount - (prf_amount + pif_amount + wakalah_fee_shareholders_fund)) > 0.01
""").show()


┌─────────────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────────────────────┬───────────────┐
│ contribution_id │ policy_id │ gross_amount  │  prf_amount   │  pif_amount   │ wakalah_fee_shareholders_fund │     diff      │
│     varchar     │  varchar  │ decimal(12,2) │ decimal(12,2) │ decimal(12,2) │         decimal(12,2)         │ decimal(15,2) │
└─────────────────┴───────────┴───────────────┴───────────────┴───────────────┴───────────────────────────────┴───────────────┘
                                                            0 rows                                                           



## PIF leakage check

`pif_amount` must be 0 wherever the policy has no PIF component. Should return 0 rows.


In [3]:
con.sql("""
    select c.*
    from stg_contributions c
    join stg_policies p on c.policy_id = p.policy_id
    where p.has_pif = false and c.pif_amount != 0
""").show()


┌─────────────────┬───────────┬───────────────────┬───────────────┬───────────────────────────────┬───────────────┬───────────────┬────────────────┐
│ contribution_id │ policy_id │ contribution_date │ gross_amount  │ wakalah_fee_shareholders_fund │  prf_amount   │  pif_amount   │ payment_method │
│     varchar     │  varchar  │       date        │ decimal(12,2) │         decimal(12,2)         │ decimal(12,2) │ decimal(12,2) │    varchar     │
└─────────────────┴───────────┴───────────────────┴───────────────┴───────────────────────────────┴───────────────┴───────────────┴────────────────┘
                                                                       0 rows                                                                     



## Open DuckDB's built-in web UI

Per DuckDB's own docs: *"running the UI directly on a read-only database is not
supported... run the UI on another [writable] database"* — this needs an actual
writable **file**, not an in-memory catalog (tried that first; still failed the
same way once opened in the browser).

`ui_con` connects to a small dedicated file (`duckdb_data/ui_catalog.duckdb`,
gitignored, holds only UI state/history — never touched by dbt/load.py/Airflow),
then `ATTACH`es the real data read-only and `USE`s it so bare table names work in
the UI's query editor. Our actual data stays exactly as protected as `con` above.

**Caveat:** `ui_con` itself stays a separate, harmless writable file regardless —
but the read-only `ATTACH` still means: don't run `dbt run`/`dbt test`/Airflow while
this UI tab is open, for the same single-writer reason as `con`.


In [2]:
ui_con = duckdb.connect('../duckdb_data/ui_catalog.duckdb')
ui_con.sql("ATTACH '../duckdb_data/takaful_transform.duckdb' AS takaful (READ_ONLY)")
ui_con.sql("USE takaful")
ui_con.sql("LOAD httpfs;")
ui_con.sql("""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_url_style='path';
    SET s3_use_ssl=false;
""")
ui_con.sql("INSTALL ui; LOAD ui;")
ui_con.sql("CALL start_ui();")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┐
│                result                │
│               varchar                │
├──────────────────────────────────────┤
│ UI started at http://localhost:4213/ │
└──────────────────────────────────────┘